# Learning Curve Diagnosis

This notebook demonstrates how to use the `get_learning_curve_data` method to diagnose underfitting vs. overfitting in the quantile models.

**Goal:** Determine if the model would benefit from more data (high variance/overfitting) or needs more complex features/models (high bias/underfitting).

In [ ]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.db.client import get_engine
from src.models.feature_store import FeatureStore
from src.models.quantile_trainer import QuantileModelSuite, QuantileModelConfig

# Optional: Import plotting library if available
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_style("whitegrid")
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    print("Matplotlib/Seaborn not found. Tables will be displayed instead.")

## 1. Load Data

We load training data for the recent seasons.

In [ ]:
engine = get_engine()
store = FeatureStore(engine)

train_seasons = ["22022", "22023"]
df = store.get_training_dataset(train_seasons)

print(f"Loaded {len(df):,} rows")

## 2. Configure Model and Features

Select a specific stat (e.g., Points) and define the features.

In [ ]:
TARGET_STAT = "pts"
TARGET_COL = f"{TARGET_STAT}_per_min"

# Define features (using a subset of standard features for this demo)
FEATURES = [
    "player_avg_pts_per_min_l5",
    "player_avg_pts_per_min_l15",
    "opp_def_rating_l10",
    "is_home",
    "rest_days"
]

# Ensure features exist in DF
available_features = [f for f in FEATURES if f in df.columns]
print(f"Using features: {available_features}")

# Filter to valid rows (min minutes threshold)
mask = (df["actual_minutes"] >= 10) & (df[TARGET_COL].notna())
data_subset = df[mask].copy()

X = data_subset[available_features].fillna(0)
y = data_subset[TARGET_COL]

## 3. Run Learning Curve Diagnosis

We will train models on 10%, 30%, 50%, 80%, and 100% of the training data.

In [ ]:
config = QuantileModelConfig(
    quantiles=(0.10, 0.50, 0.90),  # Focus on key quantiles for diagnosis
    n_estimators=100,              # Reduced for speed in demo
    max_depth=4
)

suite = QuantileModelSuite(config)

# Run learning curve generation
results = suite.get_learning_curve_data(
    X, y, 
    train_sizes=[0.1, 0.3, 0.5, 0.8, 1.0]
)

## 4. Visualize Results

In [ ]:
def plot_learning_curve(quantile, data):
    df_res = pd.DataFrame(data)
    
    print(f"\n--- Learning Curve Data: Quantile {quantile} ---")
    print(df_res[["fraction", "train_size", "train_coverage", "val_coverage"]].round(4))
    
    if HAS_PLOT:
        plt.figure(figsize=(10, 6))
        plt.plot(df_res["train_size"], df_res["train_coverage"], 'o-', label='Train Coverage')
        plt.plot(df_res["train_size"], df_res["val_coverage"], 's-', label='Validation Coverage')
        
        # Target line
        plt.axhline(y=quantile, color='r', linestyle='--', label=f'Target ({quantile})')
        
        plt.xlabel("Training Set Size")
        plt.ylabel("Coverage (Proportion)")
        plt.title(f"Learning Curve: Quantile {quantile}")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

# Display for each quantile
for q, data in results.items():
    plot_learning_curve(q, data)

## Interpretation Guide

1.  **High Variance (Overfitting):**
    *   Train Coverage is perfect (or very close to target).
    *   Validation Coverage is far from target.
    *   *Solution:* Add more data, increase regularization (lower max_depth, higher min_child_weight), or reduce features.

2.  **High Bias (Underfitting):**
    *   Both Train and Validation Coverage are far from target.
    *   Adding more data doesn't help much.
    *   *Solution:* Increase model complexity (higher max_depth), add better features, or relax regularization.